# Notebook 04: Embedding EDA


Self-contained exploratory analysis of embedding geometry for both ESM-2 and AbLang2.
Covers sequence-level and residue-level delta embeddings.
All cached tensors are loaded from Drive -- no model inference in this notebook.

Analyses:
- Discriminability gain from delta computation (CoV raw vs delta)
- Delta norm distributions by dataset and by CDR/FR region
- Spearman(delta norm, DMS score) per dataset for both models and both levels
- PCA structure of delta embeddings (chain identity, region, dataset)
- Cross-model comparison: do ESM-2 and AbLang2 agree on mutation magnitude?
- Summary Spearman table: both models x both levels x 5 datasets


## Setup


In [1]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")


Environment: local
Repo: /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project


Detects whether the notebook is running on Colab or locally. On Colab, mounts Drive
and clones (or pulls) the repo. Locally, resolves the repo root from the notebook's
location in `notebooks/`.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.


In [2]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:    {DRIVE_ROOT}")
print(f"Embedding dir: {EMBEDDING_DIR}")
print(f"Figures dir:   {FIGURES_DIR}")
print("Paths set.")


Drive root:    /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project
Embedding dir: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/embeddings
Figures dir:   /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures
Paths set.


`src/config.py` resolves `DRIVE_ROOT` automatically: Colab mount path first, then
Google Drive Desktop glob (any account), then `outputs/` fallback. No per-collaborator
edits needed. The figures dir is where all plots from this notebook are saved.

Expected output: paths ending in `DL_Final_Project/Antibody_Project/...` (Drive) or
`outputs/...` (local fallback).


In [3]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")


Local run -- installation skipped.


In [4]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")


Autoreload enabled.


In [5]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")


Device: mps
Apple MPS -- Apple Silicon unified memory


Device check. This notebook does not load any models or run GPU inference --
all computations (norm, PCA, Spearman) run on CPU. The device check is included
for consistency with other notebooks.


## Imports


In [6]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.decomposition import PCA

from src.config import DATA_DIR, EMBEDDING_DIR, FIGURES_DIR
from src.visualization.plots import (
    plot_delta_norm_by_dataset,
    plot_delta_norm_cdr_vs_fr,
    plot_delta_pca,
    plot_delta_variance_ratio,
    plot_model_comparison_norms,
    plot_dms_score_distributions,
)

print("Imports OK.")

Imports OK.


## Load Data


In [7]:
# Load mutation metadata: region labels, DMS scores, dataset names, chain assignments
df = pd.read_csv(DATA_DIR / 'abagym_antibody.csv', dtype={'site': str})

regions   = df['region'].tolist()                          # 5318 strings: CDR_H1, ..., FR
dms_scores = df['MinMax_normalized_DMS_score'].values      # (5318,) float
dms_names  = df['DMS_name'].tolist()                       # 5318 strings: dataset per row
chains     = df['chains'].tolist()                         # 5318 strings: 'H' or 'L'

# Binary CDR/FR label (collapses all 6 CDR subtypes)
cdr_fr = ['FR' if r == 'FR' else 'CDR' for r in regions]

datasets = sorted(df['DMS_name'].unique())
dms_arr  = np.array(dms_names)

print(f"Rows: {len(df)}")
print(f"Datasets: {datasets}")
print(f"Region counts:")
print(df['region'].value_counts().to_string())


Rows: 5318
Datasets: ['Ang2_2017_G6', 'EGFR_2013_Cetuximab', 'HER2_2021_trastuzumab', 'VEGF_2017b_G6', 'lysozyme_2019_D441']
Region counts:
region
FR        2188
CDR_H3     851
CDR_L3     646
CDR_H2     558
CDR_L1     433
CDR_H1     415
CDR_L2     227


Loads `abagym_antibody.csv` for the per-mutation metadata needed throughout the EDA.
The `region` column was pre-computed during NB01 via ANARCI IMGT mapping and identifies
which CDR loop or framework region each mutation falls in.

Binary `cdr_fr` collapses CDR_H1 through CDR_L3 into a single 'CDR' label for the
two-group CDR vs FR comparisons.

**Confirmed output (2026-04-02):**
- Rows: 5318
- Datasets: Ang2_2017_G6, EGFR_2013_Cetuximab, HER2_2021_trastuzumab, VEGF_2017b_G6, lysozyme_2019_D441
- Region counts: FR=2188, CDR_H3=851, CDR_L3=646, CDR_H2=558, CDR_L1=433, CDR_H1=415, CDR_L2=227
- Total CDR: 3130 (58.9%) | FR: 2188 (41.1%)


In [8]:
# Load all cached delta tensors and raw embeddings (CPU only -- no model inference)
esm2_seq_raw    = torch.load(EMBEDDING_DIR / 'esm2_abagym.pt',                  map_location='cpu')
esm2_seq_delta  = torch.load(EMBEDDING_DIR / 'esm2_abagym_delta.pt',            map_location='cpu')
esm2_res_delta  = torch.load(EMBEDDING_DIR / 'esm2_abagym_residue_delta.pt',    map_location='cpu')

abl_seq_raw     = torch.load(EMBEDDING_DIR / 'ablang2_abagym.pt',               map_location='cpu')
abl_seq_delta   = torch.load(EMBEDDING_DIR / 'ablang2_abagym_delta.pt',         map_location='cpu')
abl_res_delta   = torch.load(EMBEDDING_DIR / 'ablang2_abagym_residue_delta.pt', map_location='cpu')

print(f"ESM-2  raw sequence:      {esm2_seq_raw.shape}")
print(f"ESM-2  sequence delta:    {esm2_seq_delta.shape}")
print(f"ESM-2  residue delta:     {esm2_res_delta.shape}")
print(f"AbLang2 raw sequence:     {abl_seq_raw.shape}")
print(f"AbLang2 sequence delta:   {abl_seq_delta.shape}")
print(f"AbLang2 residue delta:    {abl_res_delta.shape}")


ESM-2  raw sequence:      torch.Size([5318, 2560])
ESM-2  sequence delta:    torch.Size([5318, 2560])
ESM-2  residue delta:     torch.Size([5318, 1280])
AbLang2 raw sequence:     torch.Size([5318, 960])
AbLang2 sequence delta:   torch.Size([5318, 960])
AbLang2 residue delta:    torch.Size([5318, 480])


Loads 6 tensors from Drive. Raw embeddings (mutant sequence-level) are needed only
for the CoV variance ratio comparison. All delta tensors were computed in NB03:
delta = mutant_embedding - wildtype_embedding, aligned row-by-row.

**Confirmed shapes (2026-04-02):**
- ESM-2 raw sequence: (5318, 2560)
- ESM-2 sequence delta: (5318, 2560)
- ESM-2 residue delta: (5318, 1280)
- AbLang2 raw sequence: (5318, 960)
- AbLang2 sequence delta: (5318, 960)
- AbLang2 residue delta: (5318, 480)


## Sequence-Level EDA

Analyzes the 2560-dim (ESM-2) and 960-dim (AbLang2) sequence-level delta embeddings.
Sequence-level delta = mean_pool(mutant) - mean_pool(wildtype), concatenated over H and L chains.


In [9]:
# Compute L2 norms of sequence-level delta embeddings
esm2_seq_norms = torch.norm(esm2_seq_delta.float(), dim=1).numpy()
abl_seq_norms  = torch.norm(abl_seq_delta.float(),  dim=1).numpy()

print(f"ESM-2  sequence delta norms: min={esm2_seq_norms.min():.4f}, "
      f"median={np.median(esm2_seq_norms):.4f}, max={esm2_seq_norms.max():.4f}")
print(f"AbLang2 sequence delta norms: min={abl_seq_norms.min():.4f}, "
      f"median={np.median(abl_seq_norms):.4f}, max={abl_seq_norms.max():.4f}")


ESM-2  sequence delta norms: min=0.0254, median=0.0872, max=0.3167
AbLang2 sequence delta norms: min=0.0437, median=0.1084, max=0.6065


L2 norm of each row of the delta tensor measures how much the model's representation
changed in response to the mutation. A larger norm = larger perturbation in embedding space.
This is the core quantity used throughout the EDA.

**Confirmed output (2026-04-02):**
- ESM-2 sequence delta norms: min=0.0254, median=0.0872, max=0.3167
- AbLang2 sequence delta norms: min=0.0437, median=0.1084, max=0.6065

Notable: AbLang2 has a higher median (0.1084 vs 0.0872) and a notably wider range
(max 0.6065 vs 0.3167). The larger AbLang2 maximum suggests some mutations produce
substantially larger perturbations in the antibody-specific representation space
than in ESM-2. Whether this reflects true biological signal or model-specific
geometry is examined in the CDR/FR and Spearman analyses below.


### Discriminability Gain: Raw vs Delta CoV

Coefficient of variation (std / mean of L2 norms) measures how spread out the norm
distribution is relative to its center. Raw sequence embeddings cluster tightly around
the wildtype (high cosine similarity, low CoV). Delta embeddings amplify the mutation-
specific signal. Prior ESM-2 finding: delta CoV is 177x-301x higher than raw CoV.


In [10]:
plot_delta_variance_ratio(esm2_seq_raw, esm2_seq_delta, dms_names, 'ESM-2',   FIGURES_DIR)
plot_delta_variance_ratio(abl_seq_raw,  abl_seq_delta,  dms_names, 'AbLang2', FIGURES_DIR)


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_variance_ratio_esm2.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_variance_ratio_ablang2.png


Left panel: raw CoV vs delta CoV per dataset on a log scale.
Right panel: ratio (delta CoV / raw CoV) per dataset.

**Confirmed output (2026-04-02):**

| Dataset | ESM-2 ratio | AbLang2 ratio |
|---|---|---|
| Ang2_2017_G6 | 239x | 96x |
| EGFR_2013_Cetuximab | 223x | 105x |
| HER2_2021_trastuzumab | 302x | 91x |
| VEGF_2017b_G6 | 225x | 178x |
| lysozyme_2019_D441 | 178x | 117x |
| **Range** | **178x–302x** | **91x–178x** |

ESM-2 gains (178x–302x) are larger than AbLang2 (91x–178x). This does not mean ESM-2's
deltas are more informative -- it means AbLang2's raw embeddings are already less
compressed around wildtype, because OAS training has exposed AbLang2 to far more
antibody-specific sequence diversity.

Notably, the HER2 pattern is **reversed** between models:
- ESM-2: HER2 is the **highest** ratio (302x)
- AbLang2: HER2 is the **lowest** ratio (91x)

All 184 HER2 mutations are in CDR H3. The high ESM-2 ratio is driven by an extremely
low raw CoV denominator: ESM-2's raw mean-pooled embeddings for the 184 HER2 mutants
have nearly identical L2 norms. CDR H3 is severely under-represented in UniRef50
(antibodies are a small fraction of the proteome; 50% identity clustering further
collapses CDR H3 diversity). ESM-2 therefore assigns relatively uniform contextual
embeddings at CDR H3 positions regardless of amino acid identity -- the flanking
conserved IMGT framework context dominates the mean-pooled representation. ESM-2 is
**insensitive** to CDR H3 amino acid variation in raw embedding space, which also
manifests in the delta space where CDR H3 produces ESM-2's smallest delta norms of any
loop (Finding below). The raw CoV is compressed and the ratio is inflated as a result.

AbLang2, trained on OAS where CDR H3 is the most hypervariable loop in the repertoire,
treats CDR H3 amino acid identity as highly informative. Its raw embeddings at CDR H3
positions vary meaningfully with amino acid identity, giving a higher raw CoV and
therefore a lower ratio (91x).

This inversion is a fingerprint of domain specificity: ESM-2 is insensitive to CDR H3
variation because it has barely encountered it; AbLang2 is sensitive to it because it
has trained on it extensively.

### Delta Norm Distributions by Dataset

Violin plots of delta norms per dataset. Verifies that the norm distributions differ
meaningfully across antibodies -- if all five distributions are identical, the model
is not encoding dataset-specific structure.


In [11]:
plot_delta_norm_by_dataset(esm2_seq_delta, dms_names, 'ESM-2',   FIGURES_DIR)
plot_delta_norm_by_dataset(abl_seq_delta,  dms_names, 'AbLang2', FIGURES_DIR)


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_norm_by_dataset_esm2.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_norm_by_dataset_ablang2.png


Each panel is one AbAgym dataset, showing the distribution of delta norms for all
mutations in that dataset.

**Confirmed output (2026-04-02):**

**AbLang2 (top figure):**
- All 5 datasets have nearly identical medians (~0.09–0.10), suggesting AbLang2 responds
  with similar average magnitude across antibodies regardless of CDR/FR composition.
- Most datasets show strong right skew with long upper tails (outlier mutations reaching
  0.33–0.60), indicating a subset of mutations produce unusually large perturbations.
- HER2 is the notable exception: its distribution is bell-shaped and approximately
  symmetric, with a tight spread (~0.07–0.15). Unlike all other datasets it has no long
  upper tail. Since HER2 mutations are 100% CDR H3, this may reflect AbLang2's strong
  CDR H3 prior from OAS training -- CDR H3 mutations are routine to AbLang2, producing
  more uniform, moderate-magnitude responses with fewer extremes.

**ESM-2 (bottom figure):**
- More variation in median across datasets. HER2 has the clearly lowest median (~0.065)
  and tightest spread. Lysozyme has the highest median (~0.10) and widest spread.
- All datasets are right-skewed but less dramatically than AbLang2.
- ESM-2's Y-axis reaches ~0.32 vs AbLang2's ~0.60 -- AbLang2 produces more extreme
  outlier mutations, consistent with the wider global norm range seen in the summary stats.
- The ordering of medians by dataset (lysozyme > others > HER2 for ESM-2) is consistent
  with the inverse CDR prior: lysozyme is 66% FR mutations, which ESM-2 treats as unusual
  perturbations; HER2 is 100% CDR H3, which ESM-2 encodes as smaller shifts.


### CDR vs FR Delta Norms (the Inverse CDR Prior)

The CDR prior states that CDR mutations should have larger functional effects than
framework mutations -- CDRs are the binding interface, FR is structural scaffold.
ESM-2 encodes the INVERSE: FR mutations produce larger delta norms than CDR mutations
(Mann-Whitney p=2.78e-122). This is because ESM-2 is trained on general proteins where
FR positions are highly conserved -- any mutation there is unusual and produces a large
representation shift.

Key open question for AbLang2: does antibody-specific pretraining on OAS produce a
different structure? AbLang2 has seen far more CDR diversity than ESM-2, so it may
not be as surprised by CDR mutations.


In [12]:
plot_delta_norm_cdr_vs_fr(esm2_seq_delta, regions, 'ESM-2',   FIGURES_DIR)
plot_delta_norm_cdr_vs_fr(abl_seq_delta,  regions, 'AbLang2', FIGURES_DIR)


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_norm_cdr_vs_fr_esm2.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_norm_cdr_vs_fr_ablang2.png


Left panel: CDR vs FR violin with Mann-Whitney annotation and median values.
Right panel: all 7 region types to show CDR loop-level variation.

**Confirmed output (2026-04-02) -- ESM-2 sequence level:**

**Binary CDR vs FR (left panel):**
- CDR median: 0.0802 | FR median: 0.1022
- Direction: **FR > CDR** -- confirms the inverse CDR prior
- Mann-Whitney p=0.00e+00 (float underflow artifact, p is effectively 0, unambiguously significant)
- Effect size: FR median is **27% higher** than CDR median (ratio = 1.275x)

**7-region breakdown (right panel) -- approximate medians:**
| Region | Approx. median |
|---|---|
| CDR_H1 | ~0.104 |
| FR | ~0.102 |
| CDR_L1 | ~0.090 |
| CDR_H2 | ~0.082 |
| CDR_L3 | ~0.078 |
| CDR_L2 | ~0.075 |
| CDR_H3 | ~0.075 |

CDR_H1 is an outlier: its median (~0.104) is essentially equal to FR (~0.102). CDR_H1
adopts canonical conformations (a small set of known loop structures), making it more
structurally constrained than CDR_H3. ESM-2 therefore treats mutations in CDR_H1 as
nearly as unusual as FR mutations. The inverse CDR prior is strongest for CDR_H3 and
CDR_L2 (furthest below FR) and absent for CDR_H1.

---

**Confirmed output (2026-04-02) -- AbLang2 sequence level:**

**Binary CDR vs FR (left panel):**
- CDR median: 0.1030 | FR median: 0.1180
- Direction: **FR > CDR** -- the inverse CDR prior persists in AbLang2
- Mann-Whitney p=0.00e+00 (same float underflow artifact, unambiguously significant)
- Effect size: FR median is **14.5% higher** than CDR median (ratio = 1.145x)

**Comparison to ESM-2:** The inverse prior is attenuated but not eliminated. ESM-2's
FR/CDR ratio is 1.275x vs AbLang2's 1.145x -- antibody-specific pretraining on OAS
has partially corrected the bias but the same direction holds. The residual inverse
prior in AbLang2 may reflect genuinely unusual FR mutations (structural disruptions)
rather than a systematic training artifact.

**7-region breakdown (right panel) -- approximate medians:**
| Region | Approx. median |
|---|---|
| FR | ~0.118 |
| CDR_L3 | ~0.108 |
| CDR_L1 | ~0.107 |
| CDR_L2 | ~0.105 |
| CDR_H1 | ~0.104 |
| CDR_H3 | ~0.100 |
| CDR_H2 | ~0.098 |

Two key differences from ESM-2's 7-loop breakdown:

1. **CDR_H3 is no longer the lowest loop.** In ESM-2, CDR_H3 had the lowest median
   (~0.075) because ESM-2 has seen CDR_H3 diversity in general proteins. In AbLang2,
   CDR_H3 is mid-range (~0.100) and CDR_H2 is the lowest (~0.098). This directly
   reflects OAS training: AbLang2 has seen extensive CDR_H3 variation in antibodies,
   so those mutations are not unusual to it.

2. **CDR loops are far more homogeneous.** AbLang2's CDR medians span ~0.098–0.108
   (range 0.010), vs ESM-2's ~0.075–0.104 (range 0.029). The antibody-specific model
   responds more uniformly across CDR loops, having learned that diversity is expected
   throughout the paratope.

3. **Light chain CDRs (L1, L2, L3) have slightly higher medians** than heavy chain
   CDRs (H1, H2, H3) in AbLang2. This may reflect AbLang2's joint VH|VL encoding --
   the cross-chain attention could be shifting the light chain CDR representations
   relative to what a single-chain model would produce.

4. **FR's extreme upper tail is the main driver** of the FR > CDR result in AbLang2.
   The FR distribution extends to ~0.5 while CDR distributions top out around 0.28–0.35.
   In ESM-2 the bulk distributions were more clearly separated; in AbLang2 the CDR and FR
   bulk overlap more, and it is the FR outlier tail that pulls the FR median above CDR.

**Implication for Experiment 7 (CDR constraint):**
- ESM-2: strong inverse prior (1.275x), constraint directly opposes model geometry
- AbLang2: weaker inverse prior (1.145x), model geometry is less misaligned with the constraint
- The constraint may hurt ESM-2 more than AbLang2, or may need a different lambda range


### Spearman(delta norm, DMS score) -- Sequence Level

Measures how well the L2 norm of the delta embedding correlates with the experimental
mutation effect score (MinMax-normalized DMS). A positive Spearman r means larger
embedding perturbations correspond to larger functional effects.

Prior ESM-2 result (from NB02.5): r = 0.079 to 0.151 per dataset. This is modest but
consistent, and is the baseline signal available without any trained model on top.


In [13]:
print(f"{'Model':<10} {'Dataset':<35} {'r':>7} {'p':>10}")
print('-' * 65)

seq_spearman = {}
for ds in datasets:
    mask = dms_arr == ds
    for model, norms in [('ESM-2', esm2_seq_norms), ('AbLang2', abl_seq_norms)]:
        r, p = spearmanr(norms[mask], dms_scores[mask])
        seq_spearman[(model, ds)] = r
        print(f"{model:<10} {ds:<35} {r:>7.4f} {p:>10.2e}")
    print()

# Aggregate (all 5318 rows)
for model, norms in [('ESM-2', esm2_seq_norms), ('AbLang2', abl_seq_norms)]:
    r, p = spearmanr(norms, dms_scores)
    seq_spearman[(model, 'ALL')] = r
    print(f"{model:<10} {'ALL':<35} {r:>7.4f} {p:>10.2e}")


Model      Dataset                                   r          p
-----------------------------------------------------------------
ESM-2      Ang2_2017_G6                         0.1086   6.57e-04
AbLang2    Ang2_2017_G6                         0.3174   2.16e-24

ESM-2      EGFR_2013_Cetuximab                  0.0788   9.88e-03
AbLang2    EGFR_2013_Cetuximab                  0.1484   1.08e-06

ESM-2      HER2_2021_trastuzumab                0.1270   8.59e-02
AbLang2    HER2_2021_trastuzumab                0.1234   9.50e-02

ESM-2      VEGF_2017b_G6                        0.1510   1.86e-06
AbLang2    VEGF_2017b_G6                        0.3753   2.15e-34

ESM-2      lysozyme_2019_D441                   0.1371   2.95e-10
AbLang2    lysozyme_2019_D441                   0.2116   1.25e-22

ESM-2      ALL                                  0.0834   1.13e-09
AbLang2    ALL                                  0.2157   4.99e-57


Per-dataset Spearman correlation between delta norm (L2 magnitude) and MinMax-normalized
DMS score. This is the unsupervised signal available from the embedding geometry alone,
before any trained model is applied. A positive r means larger embedding perturbations
correspond to larger functional effects.

**Confirmed output (2026-04-02) -- sequence level:**

| Dataset | ESM-2 r | ESM-2 p | AbLang2 r | AbLang2 p |
|---|---|---|---|---|
| Ang2_2017_G6 | 0.1086 | 6.57e-04 | 0.3174 | 2.16e-24 |
| EGFR_2013_Cetuximab | 0.0788 | 9.88e-03 | 0.1484 | 1.08e-06 |
| HER2_2021_trastuzumab | 0.1270 | 8.59e-02 | 0.1234 | 9.50e-02 |
| VEGF_2017b_G6 | 0.1510 | 1.86e-06 | 0.3753 | 2.15e-34 |
| lysozyme_2019_D441 | 0.1371 | 2.95e-10 | 0.2116 | 1.25e-22 |
| **ALL** | **0.0834** | 1.13e-09 | **0.2157** | 4.99e-57 |

Key observations:

1. **AbLang2 is substantially stronger than ESM-2 on every dataset except HER2.**
   Aggregate Spearman: AbLang2 0.2157 vs ESM-2 0.0834 -- AbLang2 is 2.6x higher.
   Per-dataset gains: Ang2 (3x), VEGF (2.5x), EGFR (1.9x), lysozyme (1.5x).

2. **HER2 is the sole exception.** Both models produce nearly identical and marginally
   non-significant correlations (ESM-2 r=0.1270, p=0.086; AbLang2 r=0.1234, p=0.095).
   Neither achieves p<0.05. This is consistent with prior expectations: HER2 has a
   bimodal DMS score distribution and all 184 mutations in CDR H3, making it the most
   difficult dataset for norm-based prediction in both models.

3. **Ang2 and VEGF are the strongest datasets for AbLang2** (r=0.3174 and r=0.3753).
   Both are G6-scaffold antibodies with 79% CDR mutations. The high correlation may
   reflect AbLang2's better encoding of CDR mutation effects due to OAS training.

4. **ESM-2 results are consistent with prior NB02.5 findings** (r=0.079–0.151 reported
   there). Current values fall within that range for all datasets.

5. **This is a norm-only correlation** -- the L2 magnitude of the delta, a single scalar
   per mutation. The trained MLP (Experiments 2–6) operates on the full 2560/960-dim
   delta vector and is expected to achieve substantially higher correlations by learning
   directional structure in the embedding space, not just magnitude.

The strength of AbLang2's unsupervised signal (r=0.38 for VEGF before any training)
suggests that AbLang2's delta geometry is more directly aligned with functional mutation
effects than ESM-2's, likely because OAS training has encoded antibody-specific
sequence-function relationships that ESM-2's general protein training has not.


### PCA Structure -- Sequence Level

PCA of delta embeddings reveals geometric structure in the high-dimensional space.

Known ESM-2 finding: PC1 and PC2 form an orthogonal cross, where PC1 separates heavy
chain mutations and PC2 separates light chain mutations. This cross arises because
ESM-2 embeds H and L in separate forward passes -- the delta affects only one half
of the 2560-dim concatenated vector, producing orthogonal perturbation directions.

AbLang2 hypothesis: cross-chain attention mixes the H/L representations in a single
forward pass, so the orthogonal cross structure may not appear. Whether the H/L
separation is visible in a different form (or absent) is an open question.


In [14]:
# PCA colored by chain (H vs L) -- tests for orthogonal H/L subspace structure
plot_delta_pca(esm2_seq_delta, np.array(chains), 'chain', 'ESM-2',   FIGURES_DIR, 'sequence')
plot_delta_pca(abl_seq_delta,  np.array(chains), 'chain', 'AbLang2', FIGURES_DIR, 'sequence')


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_pca_esm2_chain_sequence.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_pca_ablang2_chain_sequence.png


PCA scatter colored by chain identity (H=blue, L=orange).

**Confirmed output (2026-04-02):**

**ESM-2 (top):**
- PC1 (18.3%) + PC2 (7.8%) = 26.1% variance explained
- Perfect orthogonal cross shape. H mutations (blue) lie entirely on the PC1 axis
  (PC2 ≈ 0). L mutations (orange) lie entirely on the PC2 axis (PC1 ≈ 0). No mixing.
- H arm extends from ~(-0.28, 0) to ~(0.12, 0). L arm extends from ~(0, -0.20) to
  ~(0, 0.12). The two arms are perpendicular to machine precision.
- Mechanism: ESM-2 embeds H and L chains in separate forward passes. The 2560-dim
  sequence delta is concat(H_delta, L_delta). For an H-chain mutation, L_delta = 0
  (unchanged), so the perturbation lives entirely in the first 1280 dims. For an L-chain
  mutation, H_delta = 0 and the perturbation lives entirely in the last 1280 dims. PCA
  identifies these two orthogonal subspaces as PC1 and PC2. The cross is not an
  emergent property of the model -- it is a direct geometric consequence of separate
  per-chain embedding.
- HER2 mutations (all H chain) would cluster only on the PC1 arm.

**AbLang2 (bottom):**
- PC1 (28.3%) + PC2 (14.6%) = 42.9% variance explained -- substantially higher than
  ESM-2's 26.1%, meaning AbLang2's delta structure is more concentrated in 2D.
- No cross shape. H (blue) and L (orange) mutations are fully mixed throughout the
  scatter. There is no axis along which one chain dominates.
- The bulk of mutations form a dense cluster near the origin (small PC1 and PC2).
  A long diagonal scatter extends toward large positive PC1 (~0.5) -- these correspond
  to the high-norm outlier mutations visible in the violin plots (predominantly FR).
- Mechanism: AbLang2 processes the full VH|VL sequence in a single forward pass through
  a shared transformer. Cross-chain attention means a mutation on chain H perturbs not
  only H token representations but also L token representations via attention. The
  sequence-level delta (concat of H and L mean-pooled tokens from the joint pass) is
  therefore nonzero in both halves for both H and L mutations. The orthogonal subspace
  structure cannot form when both halves of the embedding are coupled.

**Answer to open question:** Cross-chain attention in AbLang2 fully eliminates the
orthogonal H/L subspace structure seen in ESM-2. The two models encode chain identity
in fundamentally different geometric forms: ESM-2 produces chain-specific orthogonal
axes, AbLang2 produces a mixed representation with no chain-specific geometry.

This has implications for the PCA-based analysis: the ESM-2 "cross" means PC1 and PC2
each explain only one chain's variance (hence the low 26.1% total). AbLang2's mixed
representation allows PC1 and PC2 to capture variance from both chains simultaneously,
explaining more total variance (42.9%) in fewer components.


In [15]:
# PCA colored by binary CDR/FR
plot_delta_pca(esm2_seq_delta, np.array(cdr_fr), 'cdr_fr', 'ESM-2',   FIGURES_DIR, 'sequence')
plot_delta_pca(abl_seq_delta,  np.array(cdr_fr), 'cdr_fr', 'AbLang2', FIGURES_DIR, 'sequence')


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_pca_esm2_cdr_fr_sequence.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_pca_ablang2_cdr_fr_sequence.png


PCA colored by CDR (green) vs FR (red).

**Confirmed output (2026-04-02):**

**AbLang2 (top):**
- In the dense bulk cluster (PC1 < 0.1), CDR and FR are fully mixed -- no separation
  between the two groups at typical mutation magnitudes.
- The outlier scatter extending to large positive PC1 (>0.1, up to ~0.5) is almost
  entirely FR (red). Essentially all of the extreme high-PC1 points are FR mutations.
- Interpretation: AbLang2's PC1 is partly a mutation magnitude axis. Since FR mutations
  produce larger delta norms (as shown in the violin plots), they project further along
  PC1. The CDR/FR separation is not a clean 2D boundary -- it emerges as a density
  gradient where FR increasingly dominates at larger distances from the origin.
- This is geometrically consistent with the CDR/FR norm finding: the FR > CDR effect in
  AbLang2 is driven by FR's extreme outlier tail, and those outlier mutations are the
  ones visible as the isolated scatter at high PC1 values.

**ESM-2 (bottom):**
- CDR and FR appear on both arms of the cross, mixed throughout each arm.
- FR (red) tends toward the more extreme positions along each arm (farther from the
  cross center), while CDR (green) concentrates closer to the origin along each arm.
- This is consistent with the inverse CDR prior: FR mutations produce larger delta norms
  (larger displacement from origin) while CDR mutations produce smaller norms (closer to
  origin), but this magnitude gradient lies along each chain-specific axis rather than
  defining a separate axis of its own.
- CDR/FR is not a primary axis of variation in ESM-2's 2D PCA. The dominant structure
  (PC1 and PC2) is chain identity. CDR vs FR appears as a secondary gradient within
  each chain-specific arm.

**Contrast between models:**
In ESM-2, chain identity dominates the top 2 PCs and CDR/FR is a secondary gradient
along each arm. In AbLang2, chain identity has no dedicated axis (mixed throughout),
and PC1 partly encodes mutation magnitude with FR mutations clustering at higher values.
The models organize their delta embedding spaces in fundamentally different ways.


In [16]:
# PCA colored by dataset -- reveals per-antibody clustering
plot_delta_pca(esm2_seq_delta, np.array(dms_names), 'dataset', 'ESM-2',   FIGURES_DIR, 'sequence')
plot_delta_pca(abl_seq_delta,  np.array(dms_names), 'dataset', 'AbLang2', FIGURES_DIR, 'sequence')


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_pca_esm2_dataset_sequence.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/delta_pca_ablang2_dataset_sequence.png


PCA colored by dataset.

**Confirmed output (2026-04-02):**

**AbLang2 (top):**
- The extreme outlier scatter at high PC1 (>0.1, extending to ~0.5) is almost entirely
  lysozyme (purple). The combination of lysozyme being 66% FR and AbLang2 producing
  large delta norms for FR mutations makes lysozyme the dominant source of high-PC1
  outliers. This directly connects three observations: FR has the longest tail in the
  violin plot → those tail mutations project to high PC1 → lysozyme has the most FR
  mutations (1382 of its 2094 total) → lysozyme dominates the outlier scatter.
- In the moderate range (PC1 0.1–0.2), Ang2 and EGFR mutations also appear, consistent
  with those datasets having some FR mutations.
- In the dense bulk cluster (PC1 < 0.1), all 5 datasets are fully mixed with no
  dataset-specific clustering.
- HER2 (green) is nearly invisible -- N=184, tight moderate-magnitude deltas that
  project near the origin.

**ESM-2 (bottom):**
- No dataset clustering along either arm. All 5 datasets are spread evenly throughout
  both arms of the cross. Mutations from different antibodies with the same chain
  assignment fully overlap. ESM-2's delta space has no antibody-specific organization
  beyond chain identity in the top 2 PCs.
- HER2 (green) is only visible on the horizontal H arm (PC1 axis), as expected since
  all 184 HER2 mutations are on the heavy chain. HER2 concentrates near the center of
  the H arm (smaller |PC1| values), consistent with its lower median norm for ESM-2.
- Lysozyme (purple) appears throughout both arms, reflecting that it has both H and L
  chain mutations distributed across both arms of the cross.

**Implication for training:**
Neither model shows strong dataset-level clustering in the top 2 PCs, which is favorable:
the MLP will need to learn mutation-level features rather than memorizing dataset identity.
The absence of dataset clustering in ESM-2 (beyond chain identity) and in AbLang2's bulk
is consistent with using these embeddings as features for cross-dataset generalization.


## Residue-Level EDA

Analyzes the 1280-dim (ESM-2) and 480-dim (AbLang2) residue-level delta embeddings.
Residue-level delta = embedding at mutation site (mutant) - embedding at same site (wildtype).

This is the single-token representation at the exact mutation position, in contrast
to sequence-level which aggregates over the entire chain via mean pooling. Residue-level
deltas are used in Experiments 2, 3, 5, and 6.

Key difference for AbLang2: the residue embedding is extracted from a joint VH|VL
forward pass, so it encodes cross-chain context. ESM-2 residue embeddings come from
a single-chain forward pass (only the mutated chain is embedded).


In [17]:
esm2_res_norms = torch.norm(esm2_res_delta.float(), dim=1).numpy()
abl_res_norms  = torch.norm(abl_res_delta.float(),  dim=1).numpy()

print(f"ESM-2  residue delta norms:  min={esm2_res_norms.min():.4f}, "
      f"median={np.median(esm2_res_norms):.4f}, max={esm2_res_norms.max():.4f}")
print(f"AbLang2 residue delta norms: min={abl_res_norms.min():.4f}, "
      f"median={np.median(abl_res_norms):.4f}, max={abl_res_norms.max():.4f}")


ESM-2  residue delta norms:  min=0.7874, median=3.5169, max=5.7526
AbLang2 residue delta norms: min=3.2136, median=5.7217, max=7.9464


L2 norms of residue-level delta embeddings. The residue delta is the change at a single
token position (mutation site), not averaged over the full chain.

**Confirmed output (2026-04-02):**

| Metric | ESM-2 residue | AbLang2 residue | ESM-2 sequence | AbLang2 sequence |
|---|---|---|---|---|
| min | 0.7874 | 3.2136 | 0.0254 | 0.0437 |
| median | 3.5169 | 5.7217 | 0.0872 | 0.1084 |
| max | 5.7526 | 7.9464 | 0.3167 | 0.6065 |
| max/min ratio | 7.3x | 2.5x | 12.5x | 13.9x |

Residue norms are ~40x larger than sequence norms for ESM-2 (median 3.52 vs 0.087) and
~53x larger for AbLang2 (median 5.72 vs 0.108). This is expected: mean pooling over
~200 residues dilutes the single-residue perturbation, whereas the residue delta is
the concentrated change at exactly one token.

Two notable observations:

1. **AbLang2 residue norms have an extremely tight range.** The max/min ratio is only
   2.5x (3.21 to 7.95) compared to 13.9x for AbLang2 sequence-level and 7.3x for ESM-2
   residue. AbLang2's minimum residue norm (3.21) is itself large -- there are essentially
   no near-zero residue deltas. This suggests that at the single-residue level, AbLang2
   responds with a minimum baseline perturbation regardless of how conservative the
   mutation is. This may reflect the joint VH|VL encoding: even small mutations to one
   residue propagate through cross-chain attention and produce nonzero changes throughout
   the representation.

2. **ESM-2 residue norms span a wider range** (0.79 to 5.75, ratio 7.3x), with a
   non-negligible tail at low values (min 0.79 is much lower than AbLang2's 3.21).
   Some mutations produce very small single-residue perturbations in ESM-2, corresponding
   to conservative substitutions at positions where ESM-2's per-residue representation
   is relatively insensitive.


### CDR vs FR Delta Norms -- Residue Level

Tests whether the inverse CDR prior observed at sequence level is also present at
residue level. At the residue level the question is more localized: does the single-
token perturbation at the mutation site differ between CDR and FR positions?


In [18]:
# Residue-level CDR vs FR -- save with a suffix to distinguish from sequence-level figures
from src.visualization.plots import plot_delta_norm_cdr_vs_fr as _plot_cdr_fr

# We call the same function; filenames are distinguished by model_name
# To separate sequence vs residue files, temporarily patch output names
# by saving to a subdirectory
res_fig_dir = FIGURES_DIR / 'residue_level'
res_fig_dir.mkdir(parents=True, exist_ok=True)

plot_delta_norm_cdr_vs_fr(esm2_res_delta, regions, 'ESM-2',   res_fig_dir)
plot_delta_norm_cdr_vs_fr(abl_res_delta,  regions, 'AbLang2', res_fig_dir)


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_norm_cdr_vs_fr_esm2.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_norm_cdr_vs_fr_ablang2.png


Residue-level CDR vs FR comparison. Saved to `figures/residue_level/`.

**Confirmed output (2026-04-02):**

| | ESM-2 residue | AbLang2 residue |
|---|---|---|
| CDR median | 3.4586 | 5.8089 |
| FR median | 3.6193 | 5.5917 |
| Direction | **FR > CDR** | **CDR > FR** |
| FR/CDR or CDR/FR ratio | 1.046x | 1.039x |
| Mann-Whitney p | 5.85e-22 | 1.47e-26 |

**AbLang2 reverses direction at the residue level.** At sequence level, AbLang2 showed
FR > CDR (1.145x). At residue level, AbLang2 shows CDR > FR (1.039x) -- the correct
biological prior. ESM-2 shows FR > CDR at both levels (1.275x sequence, 1.046x residue),
with attenuation but no reversal.

Comparison to sequence level:

| Model | Sequence direction | Sequence ratio | Residue direction | Residue ratio |
|---|---|---|---|---|
| ESM-2 | FR > CDR | 1.275x | FR > CDR | 1.046x |
| AbLang2 | FR > CDR | 1.145x | CDR > FR | 1.039x |

**Mechanistic interpretation of the AbLang2 reversal:**
The sequence-level delta is the mean of all per-residue token changes across the chain.
When a FR position is mutated in AbLang2, the change propagates broadly through the
joint representation via cross-chain attention, affecting the mean-pooled output for the
entire chain. This global propagation makes FR sequence-level deltas large. At the
single-token level (the mutation site only), the question is different: how much does
AbLang2's embedding at that specific position change when the amino acid identity changes?

CDR positions in AbLang2 (trained on OAS) encode rich, position-specific amino acid
diversity -- many different amino acids are expected at each CDR position, and AbLang2's
per-residue representations reflect this variability. A substitution at a CDR position
produces a large change in the token embedding because the model has a strong, diverse
prior for that position. FR positions are more conserved even within the antibody
repertoire; AbLang2's FR token embeddings may have lower per-position sensitivity, even
though FR mutations propagate more broadly through the global representation.

In short: AbLang2 encodes CDR positions with higher single-token sensitivity (CDR > FR
at residue level) but FR mutations spread further through the joint representation
(FR > CDR at sequence level). ESM-2 lacks cross-chain attention and cannot produce this
propagation effect, so FR > CDR persists at both levels.

**7-loop breakdown -- AbLang2 residue (approximate medians):**
| Region | Approx. median |
|---|---|
| CDR_H3 | ~6.00 (highest loop) |
| CDR_L2 | ~5.80 |
| CDR_L3 | ~5.80 |
| CDR_H1 | ~5.60 |
| FR | ~5.60 |
| CDR_L1 | ~5.55 |
| CDR_H2 | ~5.40 (lowest CDR) |

CDR_H3 is the highest loop at the residue level (~6.00), the opposite of the sequence-
level finding where CDR_H3 was mid-range. At the single-token level, AbLang2 shows
the strongest per-position sensitivity for CDR_H3 -- consistent with OAS training
encoding the widest per-position amino acid diversity there.

**7-loop breakdown -- ESM-2 residue (approximate medians):**
| Region | Approx. median |
|---|---|
| CDR_H2 | ~3.60 |
| FR | ~3.60 |
| CDR_H3 | ~3.45 |
| CDR_H1 | ~3.40 |
| CDR_L1 | ~3.35 |
| CDR_L2 | ~3.35 |
| CDR_L3 | ~3.35 |

ESM-2 residue: CDR_H2 and FR have nearly identical medians (~3.60). All loops are
compressed into a narrow range (3.35–3.60), a much smaller spread than sequence level.
The inverse prior persists (FR ≈ CDR_H2 > others) but is substantially attenuated
at the single-token level. FR's long lower tail (extending to ~1.0) indicates that some
FR mutations produce very small per-token perturbations in ESM-2, unlike sequence level
where no FR mutation had a near-zero delta.


### Spearman(delta norm, DMS score) -- Residue Level

Residue-level Spearman tells us whether the single-position perturbation norm is a
better or worse predictor of functional effect than the sequence-level norm.
This directly informs whether residue-level embeddings (Experiments 2, 3, 5, 6)
carry useful signal before any training.


In [19]:
print(f"{'Model':<10} {'Dataset':<35} {'r':>7} {'p':>10}")
print('-' * 65)

res_spearman = {}
for ds in datasets:
    mask = dms_arr == ds
    for model, norms in [('ESM-2', esm2_res_norms), ('AbLang2', abl_res_norms)]:
        r, p = spearmanr(norms[mask], dms_scores[mask])
        res_spearman[(model, ds)] = r
        print(f"{model:<10} {ds:<35} {r:>7.4f} {p:>10.2e}")
    print()

for model, norms in [('ESM-2', esm2_res_norms), ('AbLang2', abl_res_norms)]:
    r, p = spearmanr(norms, dms_scores)
    res_spearman[(model, 'ALL')] = r
    print(f"{model:<10} {'ALL':<35} {r:>7.4f} {p:>10.2e}")


Model      Dataset                                   r          p
-----------------------------------------------------------------
ESM-2      Ang2_2017_G6                        -0.0273   3.93e-01
AbLang2    Ang2_2017_G6                         0.2661   2.34e-17

ESM-2      EGFR_2013_Cetuximab                  0.0141   6.44e-01
AbLang2    EGFR_2013_Cetuximab                  0.1318   1.51e-05

ESM-2      HER2_2021_trastuzumab                0.0090   9.04e-01
AbLang2    HER2_2021_trastuzumab                0.1426   5.35e-02

ESM-2      VEGF_2017b_G6                        0.1219   1.23e-04
AbLang2    VEGF_2017b_G6                        0.2584   1.57e-16

ESM-2      lysozyme_2019_D441                  -0.0113   6.04e-01
AbLang2    lysozyme_2019_D441                  -0.0627   4.09e-03

ESM-2      ALL                                  0.0057   6.77e-01
AbLang2    ALL                                  0.1052   1.43e-14


Residue-level Spearman per dataset for both models.

**Confirmed output (2026-04-02):**

| Dataset | ESM-2 r | ESM-2 p | AbLang2 r | AbLang2 p |
|---|---|---|---|---|
| Ang2_2017_G6 | -0.0273 | 3.93e-01 | 0.2661 | 2.34e-17 |
| EGFR_2013_Cetuximab | 0.0141 | 6.44e-01 | 0.1318 | 1.51e-05 |
| HER2_2021_trastuzumab | 0.0090 | 9.04e-01 | 0.1426 | 5.35e-02 |
| VEGF_2017b_G6 | 0.1219 | 1.23e-04 | 0.2584 | 1.57e-16 |
| lysozyme_2019_D441 | -0.0113 | 6.04e-01 | -0.0627 | 4.09e-03 |
| **ALL** | **0.0057** | 6.77e-01 | **0.1052** | 1.43e-14 |

Key observations:

1. **ESM-2 residue norm carries no predictive signal.** Aggregate r=0.0057, p=0.677
   (not significant). Only VEGF reaches significance (r=0.1219). Four of five datasets
   are non-significant and two are mildly negative (Ang2: -0.027, lysozyme: -0.011).
   At sequence level ESM-2 had aggregate r=0.0834 (p=1e-9). At residue level it
   drops to noise. The single-token perturbation norm in ESM-2 contains no useful
   scalar information about functional mutation effects.

2. **AbLang2 residue maintains partial signal** -- aggregate r=0.1052 (p=1.43e-14),
   substantially weaker than AbLang2 sequence (r=0.2157) but meaningfully above zero.
   Ang2 (r=0.266) and VEGF (r=0.258) retain strong signal; EGFR (r=0.132) moderate.

3. **Lysozyme is negative at residue level for both models** (ESM-2: -0.011, p=0.60;
   AbLang2: -0.063, p=0.004). The sequence level was positive for both (ESM-2: 0.137,
   AbLang2: 0.212). The sign reversal is mechanistically linked to the CDR/FR findings:
   at the residue level, AbLang2's CDR > FR ordering means CDR positions carry larger
   norms. Lysozyme is 66% FR, so most of its mutations are at FR positions (lower residue
   norms in AbLang2). FR mutations in lysozyme still have real functional effects -- the
   result is that larger residue norms (CDR positions) do not correspond to larger effects,
   giving a negative correlation. This is a direct consequence of AbLang2's residue-level
   CDR prior interacting with lysozyme's FR-heavy composition.

4. **HER2 residue: AbLang2 r=0.1426, p=0.054** -- borderline, as at sequence level.
   ESM-2 HER2 residue r=0.0090 (p=0.904), effectively zero.


### PCA Structure -- Residue Level

PCA of the residue-level delta embeddings (1280-dim for ESM-2, 480-dim for AbLang2).
Unlike the sequence-level case where the 2560-dim ESM-2 delta is an explicit
concatenation of H and L chain contributions, the residue-level delta is a single
1280-dim token embedding from whichever chain the mutation is on. There is no
H||L concatenation in the residue tensor.

Despite this, ESM-2 residue PCA shows a clear cross structure -- confirmed below.
The mechanism is different from the sequence-level cross: H and L mutations were
processed in separate single-chain forward passes, so their 1280-dim residue embeddings
were contextualized by different surrounding sequences. PCA identifies two dominant
directions that correspond to H-context vs L-context embeddings, producing a cross even
without explicit concatenation.

AbLang2 residue PCA is expected to show no cross, since all residue embeddings come
from the same joint VH|VL forward pass regardless of which chain the mutation is on.

In [22]:
plot_delta_pca(esm2_res_delta, np.array(chains),    'chain',   'ESM-2',   res_fig_dir, 'residue')
plot_delta_pca(abl_res_delta,  np.array(chains),    'chain',   'AbLang2', res_fig_dir, 'residue')

plot_delta_pca(esm2_res_delta, np.array(cdr_fr),    'cdr_fr',  'ESM-2',   res_fig_dir, 'residue')
plot_delta_pca(abl_res_delta,  np.array(cdr_fr),    'cdr_fr',  'AbLang2', res_fig_dir, 'residue')

plot_delta_pca(esm2_res_delta, np.array(dms_names), 'dataset', 'ESM-2',   res_fig_dir, 'residue')
plot_delta_pca(abl_res_delta,  np.array(dms_names), 'dataset', 'AbLang2', res_fig_dir, 'residue')

Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_pca_esm2_chain_residue.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_pca_ablang2_chain_residue.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_pca_esm2_cdr_fr_residue.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_pca_ablang2_cdr_fr_residue.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/residue_level/delta_pca_esm2_dataset_residue.png
Saved: /Users/oscarrodriguez/Library/Clo

Residue-level PCA colored by chain, CDR/FR, and dataset. Saved to `figures/residue_level/`.

**Confirmed output (2026-04-03) -- AbLang2 residue:**

PC1 (6.4%) + PC2 (6.1%) = **12.5% variance explained** -- dramatically lower than
AbLang2 sequence-level (42.9%) or ESM-2 sequence-level (26.1%). The residue-level
delta space is far more diffuse; the first two PCs capture very little of the structure.

Chain and CDR/FR figures show a diffuse ellipse elongated along PC1, with no clusters,
no cross shape, and no separation between groups. H and L mutations are fully mixed.
CDR and FR are fully mixed. One subtle asymmetry in the CDR/FR figure: CDR points
extend slightly further along positive PC1 and into the upper PC2 region, consistent
with the CDR > FR residue norm finding -- larger per-token delta magnitudes scatter
further from the origin. It is not a separation and does not suggest any clustering.

Dataset coloring confirms no dataset-specific regions: all 5 antibody systems are
distributed uniformly across the ellipse. The flat structure is a property of the
model's joint VH|VL forward pass, not of dataset mixing.

The low explained variance reflects the genuine high-dimensionality of residue-level
delta signals: each sample is a 480-dim token embedding change encoding position-specific
and amino acid-specific information. Mean pooling at the sequence level averages out
positional variation, concentrating the remaining signal into fewer dimensions (hence
42.9% vs 12.5% in two PCs).

---

**Confirmed output (2026-04-03) -- ESM-2 residue:**

PC1 (7.2%) + PC2 (6.6%) = **13.8% variance explained**.

Chain and CDR/FR figures show a **clear cross structure** with well-defined arms along
PC1 (horizontal) and PC2 (vertical). H and L chains are fully mixed throughout both
arms -- the cross persists from the sequence level but has lost its chain-discriminating
quality (at sequence level, each arm was nearly pure H or L). CDR and FR are also
fully mixed across both arms. A distinct outlier cluster of ~50-100 points is visible
at approximately (-2.0, 0.4), detached from the main mass.

Dataset coloring shows all 5 datasets distributed throughout the cross arms and the
outlier cluster with no dataset-specific separation. The cross geometry is therefore
a structural property of ESM-2's per-chain forward passes -- not an artifact of mixing
datasets with different mutation distributions. The outlier cluster is multi-dataset
and its origin is not resolved by any of the three colorings used here.

**Comparison summary (residue level):**

| Model   | PC1+PC2 | Structure                      | Chain | CDR/FR | Dataset |
|---------|---------|--------------------------------|-------|--------|---------|
| AbLang2 | 12.5%   | Diffuse ellipse                | Mixed | Mixed  | Mixed   |
| ESM-2   | 13.8%   | Clear cross + outlier cluster  | Mixed | Mixed  | Mixed   |

The cross in ESM-2 residue PCA is an architectural signature: per-chain forward passes
imprint chain-context geometry even at the token level, but the signal is not strong
enough to separate chains in 2D projection. AbLang2's joint VH|VL attention erases
this structure entirely.

### Sequence vs Residue: Spearman Comparison

Direct side-by-side comparison of sequence-level and residue-level Spearman values
for both models. Informs the expected difficulty of Experiments 2-6 before training.


In [21]:
print(f"{'Dataset':<35} {'ESM2-Seq':>10} {'ESM2-Res':>10} {'ABL-Seq':>10} {'ABL-Res':>10}")
print('-' * 75)

for ds in datasets + ['ALL']:
    print(
        f"{ds:<35}"
        f"{seq_spearman.get(('ESM-2', ds), float('nan')):>10.4f}"
        f"{res_spearman.get(('ESM-2', ds), float('nan')):>10.4f}"
        f"{seq_spearman.get(('AbLang2', ds), float('nan')):>10.4f}"
        f"{res_spearman.get(('AbLang2', ds), float('nan')):>10.4f}"
    )


Dataset                               ESM2-Seq   ESM2-Res    ABL-Seq    ABL-Res
---------------------------------------------------------------------------
Ang2_2017_G6                           0.1086   -0.0273    0.3174    0.2661
EGFR_2013_Cetuximab                    0.0788    0.0141    0.1484    0.1318
HER2_2021_trastuzumab                  0.1270    0.0090    0.1234    0.1426
VEGF_2017b_G6                          0.1510    0.1219    0.3753    0.2584
lysozyme_2019_D441                     0.1371   -0.0113    0.2116   -0.0627
ALL                                    0.0834    0.0057    0.2157    0.1052


Summary comparison: both models x both embedding levels x 5 datasets + aggregate.

**Confirmed output (2026-04-03):**

| Dataset | ESM-2 Seq | ESM-2 Res | AbLang2 Seq | AbLang2 Res |
|---|---|---|---|---|
| Ang2_2017_G6 | 0.1086 | -0.0273 | 0.3174 | 0.2661 |
| EGFR_2013_Cetuximab | 0.0788 | 0.0141 | 0.1484 | 0.1318 |
| HER2_2021_trastuzumab | 0.1270 | 0.0090 | 0.1234 | 0.1426 |
| VEGF_2017b_G6 | 0.1510 | 0.1219 | 0.3753 | 0.2584 |
| lysozyme_2019_D441 | 0.1371 | -0.0113 | 0.2116 | -0.0627 |
| **ALL** | **0.0834** | **0.0057** | **0.2157** | **0.1052** |

Pattern: AbLang2 > ESM-2 at both levels. Sequence > residue for both models.
ESM-2 residue is essentially uninformative as a scalar (aggregate r≈0, p=0.677).
AbLang2 residue retains roughly half its sequence-level signal (0.105 vs 0.216).

Lysozyme is the one dataset where residue norms hurt rather than help -- negative at
residue level for both models, driven by the CDR > FR residue norm ordering interacting
with lysozyme's FR-heavy composition (see residue CDR/FR section).

Implication for training: the full delta vectors (2560-dim ESM-2, 960-dim AbLang2)
contain directional information that the scalar norm cannot capture. ESM-2's near-zero
residue norm Spearman does not mean the residue vectors are uninformative -- it means the
norm alone is not a good summary. The MLP trained on the full vector may still extract
useful signal from ESM-2 residue embeddings through learned directional projections.

## Cross-Model Comparison

ESM-2 and AbLang2 are trained on different data with different architectures. Their
delta embeddings may encode the same mutation-level information (high Spearman between
norm vectors), or complementary information (low Spearman, suggesting their outputs
could be combined for better prediction).


In [23]:
r_seq, p_seq = spearmanr(esm2_seq_norms, abl_seq_norms)
r_res, p_res = spearmanr(esm2_res_norms, abl_res_norms)

print(f"Spearman(ESM-2 seq norms, AbLang2 seq norms): r={r_seq:.4f}, p={p_seq:.2e}")
print(f"Spearman(ESM-2 res norms, AbLang2 res norms): r={r_res:.4f}, p={p_res:.2e}")


Spearman(ESM-2 seq norms, AbLang2 seq norms): r=0.3617, p=4.66e-164
Spearman(ESM-2 res norms, AbLang2 res norms): r=0.0522, p=1.41e-04


Spearman correlation between ESM-2 and AbLang2 delta norm vectors across all 5318
mutations. High r (>0.5) would indicate the models largely agree on which mutations
produce large vs small embedding changes. Low r (<0.3) suggests the models encode
different aspects of mutation effect and may benefit from combination.

**Confirmed output (2026-04-03):**

| Level    | r      | p          | Interpretation         |
|----------|--------|------------|------------------------|
| Sequence | 0.3617 | 4.66e-164  | Moderate agreement     |
| Residue  | 0.0522 | 1.41e-04   | Near-zero agreement    |

**Sequence level (r=0.3617):** The models share moderate agreement on mutation
magnitude at the sequence level -- both encode some common signal about how much a
mutation shifts the antibody-wide embedding. However, r²=0.13 means only 13% of
variance is shared; the two models are substantially independent. This is consistent
with AbLang2 having 2.6x higher DMS Spearman than ESM-2 (0.216 vs 0.083): AbLang2
captures signal that ESM-2 does not, even though both partially track the same
underlying mutation magnitude axis.

**Residue level (r=0.0522):** Near-zero cross-model agreement. The models assign
essentially uncorrelated per-token delta magnitudes to the same mutations. Given that
ESM-2 residue norm Spearman with DMS is essentially zero (r=0.006) and AbLang2
residue norm Spearman is moderate (r=0.105), the near-zero cross-model agreement
reflects that ESM-2 residue norms are uninformative while AbLang2 residue norms carry
real signal. The p-value (1.41e-04) is significant only because N=5318 -- the effect
size is negligible.

**Implication for experiment design:** The low cross-model agreement at the residue
level means ESM-2 and AbLang2 residue delta vectors are largely complementary -- they
encode different per-token information about mutation effect. Concatenating both
models' residue embeddings as MLP input may capture more signal than either alone.
At the sequence level, the moderate agreement (r=0.36) means combining models will
show diminishing returns relative to the residue level combination.

In [24]:
plot_model_comparison_norms(esm2_seq_norms, abl_seq_norms, regions, 'sequence', FIGURES_DIR)
plot_model_comparison_norms(esm2_res_norms, abl_res_norms, regions, 'residue',  FIGURES_DIR)


Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/model_comparison_norms_sequence.png
Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/model_comparison_norms_residue.png


Scatter plots of ESM-2 vs AbLang2 delta norms, colored by CDR (green) vs FR (red).
The dashed diagonal is y=x -- points above it indicate AbLang2 produces a larger norm
for that mutation; points below indicate ESM-2 produces a larger norm. Saved to
`figures/`.

**Confirmed output (2026-04-03):**

**Sequence-level (r=0.3617):**
The mass of points sits below the y=x diagonal -- AbLang2 sequence-level norms are
smaller than ESM-2 norms for the majority of mutations. However, FR mutations (pink)
show a prominent upward plume: many FR points reach AbLang2 norms of 0.3-0.6 while
their ESM-2 norms remain below 0.2. CDR mutations (green) are tightly clustered near
or below the diagonal with little spread. The FR amplification is interpretable via
AbLang2's joint VH|VL attention: an FR mutation in one chain can propagate through
cross-chain attention to affect the full paired representation, producing a large
sequence-level shift. ESM-2's independent chain processing confines each FR mutation's
effect to its own chain embedding, yielding a smaller sequence-level norm. This
mechanism would selectively amplify FR sequence norms in AbLang2 but not in ESM-2,
exactly matching the observed pattern.

**Residue-level (r=0.0522):**
The cloud sits above the y=x diagonal throughout -- AbLang2 residue-level norms are
systematically larger than ESM-2 residue norms across all mutations. The cloud is
roughly circular with no discernible positive slope, consistent with the near-zero
Spearman. CDR and FR points are distributed throughout the cloud without clear
separation. The absence of any trend confirms that ESM-2 and AbLang2 assign
essentially independent per-token magnitudes to the same mutations: knowing ESM-2's
residue norm for a mutation gives no information about AbLang2's residue norm for
that mutation.

## Summary

Full Spearman table (both models x both levels x all datasets) and key findings.


In [25]:
rows = []
for ds in datasets + ['ALL']:
    rows.append({
        'Dataset':     ds,
        'ESM-2 Seq':   seq_spearman.get(('ESM-2',   ds), float('nan')),
        'ESM-2 Res':   res_spearman.get(('ESM-2',   ds), float('nan')),
        'AbLang2 Seq': seq_spearman.get(('AbLang2', ds), float('nan')),
        'AbLang2 Res': res_spearman.get(('AbLang2', ds), float('nan')),
    })

summary_df = pd.DataFrame(rows).set_index('Dataset')
print(summary_df.to_string(float_format='{:.4f}'.format))


                       ESM-2 Seq  ESM-2 Res  AbLang2 Seq  AbLang2 Res
Dataset                                                              
Ang2_2017_G6              0.1086    -0.0273       0.3174       0.2661
EGFR_2013_Cetuximab       0.0788     0.0141       0.1484       0.1318
HER2_2021_trastuzumab     0.1270     0.0090       0.1234       0.1426
VEGF_2017b_G6             0.1510     0.1219       0.3753       0.2584
lysozyme_2019_D441        0.1371    -0.0113       0.2116      -0.0627
ALL                       0.0834     0.0057       0.2157       0.1052


Full Spearman summary table: both models x both embedding levels x all datasets.
This is the key output of NB04.

**Confirmed output (2026-04-03):**

| Dataset | ESM-2 Seq | ESM-2 Res | AbLang2 Seq | AbLang2 Res |
|---|---|---|---|---|
| Ang2_2017_G6 | 0.1086 | -0.0273 | 0.3174 | 0.2661 |
| EGFR_2013_Cetuximab | 0.0788 | 0.0141 | 0.1484 | 0.1318 |
| HER2_2021_trastuzumab | 0.1270 | 0.0090 | 0.1234 | 0.1426 |
| VEGF_2017b_G6 | 0.1510 | 0.1219 | 0.3753 | 0.2584 |
| lysozyme_2019_D441 | 0.1371 | -0.0113 | 0.2116 | -0.0627 |
| **ALL** | **0.0834** | **0.0057** | **0.2157** | **0.1052** |

**Key takeaways:**

1. **AbLang2 > ESM-2 at both levels.** AbLang2 aggregate Spearman is 2.6x higher at
   sequence level (0.216 vs 0.083) and 18x higher at residue level (0.105 vs 0.006).
   Antibody-specific pretraining is clearly more informative for mutation effect
   prediction than general protein pretraining, even before any supervised fine-tuning.

2. **Sequence > residue for both models.** The scalar norm of the mean-pooled sequence
   delta is a better unsupervised predictor than the scalar norm of the per-token delta.
   ESM-2 residue norm is essentially uninformative (r=0.006, p=0.677). AbLang2 residue
   norm retains roughly half its sequence-level signal (0.105 vs 0.216).

3. **G6 scaffold datasets are the strongest signal.** Ang2 and VEGF (both G6 scaffold)
   show the highest AbLang2 Spearman (0.317 and 0.375). Shared framework likely makes
   mutation effects more consistent and easier to rank by embedding distance.

4. **Lysozyme inverts at residue level.** Both models go negative for lysozyme at
   residue level (-0.011 ESM-2, -0.063 AbLang2). Driven by the CDR > FR residue norm
   ordering interacting with lysozyme's 66% FR composition: FR mutations (lower residue
   norm under AbLang2) still have large functional effects, inverting the correlation.

5. **HER2 is a persistent outlier.** Lowest N (184), all CDR H3, bimodal DMS
   distribution. Spearman is non-significant for ESM-2 sequence level and near-zero at
   residue level. AbLang2 performs slightly better (0.123-0.143) but HER2 should be
   reported separately in all analyses.

6. **Models are largely complementary.** Cross-model Spearman: r=0.36 at sequence
   level, r=0.05 at residue level. Residue-level vectors from the two models are
   essentially independent -- concatenation may capture more signal than either alone.

## DMS Score Distributions

Histogram of MinMax-normalized DMS scores for each of the 5 datasets.
All scores are normalized to [0, 1] within their dataset, so this shows
the shape of the label distribution, not absolute fitness values.
This plot verifies the claimed bimodal distribution for HER2 and flags
any other unusual distributions before training.

In [11]:
plot_dms_score_distributions(
    np.array(dms_scores),
    np.array(dms_names),
    FIGURES_DIR,
)

Saved: /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/results/figures/dms_score_distributions.png


DMS score histograms for all 5 datasets. Saved to `figures/`.

Record confirmed output here after running.